In [3]:
# ============================================================
# CELL 1 — LOW RAM CHECK (COLAB SAFE)
# ============================================================

import os
import subprocess

print("=" * 70)
print("SYSTEM MEMORY")
print("=" * 70)

subprocess.run(
    ["free", "-h"],
    check=False
)

print()
print("=" * 70)
print("EXISTING SWAP")
print("=" * 70)

subprocess.run(
    ["swapon", "--show"],
    check=False
)

print()
print("Trying optional swap...")

swap_file = "/content/colab_swap"

# Remove stale file if it exists
if os.path.exists(swap_file):
    try:
        os.remove(swap_file)
    except Exception:
        pass

try:
    subprocess.run(
        [
            "fallocate",
            "-l",
            "8G",
            swap_file
        ],
        check=True
    )

    subprocess.run(
        [
            "chmod",
            "600",
            swap_file
        ],
        check=True
    )

    subprocess.run(
        [
            "mkswap",
            swap_file
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

    result = subprocess.run(
        [
            "swapon",
            swap_file
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    if result.returncode == 0:

        print("Swap enabled successfully.")

    else:

        print(
            "Colab does not allow swapon; continuing without swap."
        )

        print(
            result.stdout.strip()
        )

        try:
            os.remove(swap_file)
        except Exception:
            pass

except Exception as e:

    print(
        "Optional swap unavailable; continuing without swap."
    )

print()
print("=" * 70)
print("FINAL MEMORY")
print("=" * 70)

subprocess.run(
    ["free", "-h"],
    check=False
)

print()
print("Setup complete.")

SYSTEM MEMORY

EXISTING SWAP

Trying optional swap...
Colab does not allow swapon; continuing without swap.
swapon: /content/colab_swap: swapon failed: Invalid argument

FINAL MEMORY

Setup complete.


In [4]:
# ============================================================
# CELL 1 — SYSTEM SETUP
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

print("=" * 70)
print("HUNYUAN3D-2.1 COLAB SETUP")
print("=" * 70)

print("System Python:")
print(sys.version)

# GPU
subprocess.run(
    ["nvidia-smi"],
    check=False
)

# System packages
subprocess.run(
    [
        "apt-get",
        "update",
        "-qq"
    ],
    check=True
)

subprocess.run(
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "git",
        "wget",
        "curl",
        "ffmpeg",
        "build-essential",
        "cmake",
        "ninja-build",
        "libgl1",
        "libglib2.0-0",
        "libjpeg-dev",
        "pkg-config",
        "python3.10",
        "python3.10-dev",
        "python3.10-venv"
    ],
    check=True
)

print("\nPython 3.10 system package ready.")

HUNYUAN3D-2.1 COLAB SETUP
System Python:
3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]

Python 3.10 system package ready.


In [5]:
# ============================================================
# CELL 2 — CREATE PYTHON 3.10 VENV
# ============================================================

import shutil
import subprocess
from pathlib import Path

ENV_DIR = Path(
    "/content/hunyuan21_env"
)

PYTHON = ENV_DIR / "bin" / "python"
PIP = ENV_DIR / "bin" / "pip"

# ล้างเฉพาะ environment เก่า
if ENV_DIR.exists():
    print("Removing old environment...")
    shutil.rmtree(
        ENV_DIR,
        ignore_errors=True
    )

print("Creating Python 3.10 environment...")

subprocess.run(
    [
        "python3.10",
        "-m",
        "venv",
        str(ENV_DIR)
    ],
    check=True
)

subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "-U",
        "pip",
        "setuptools",
        "wheel"
    ],
    check=True
)

print()
print(
    subprocess.check_output(
        [
            str(PYTHON),
            "--version"
        ],
        text=True
    ).strip()
)

print("PYTHON =", PYTHON)
print("PIP    =", PIP)

Removing old environment...
Creating Python 3.10 environment...

Python 3.10.12
PYTHON = /content/hunyuan21_env/bin/python
PIP    = /content/hunyuan21_env/bin/pip


In [6]:
# ============================================================
# CELL 3 — CLONE OFFICIAL HUNYUAN3D-2.1
# ============================================================

import subprocess
from pathlib import Path

REPO = Path(
    "/content/Hunyuan3D-2.1"
)

if REPO.exists():

    print("Repository exists.")

    subprocess.run(
        [
            "git",
            "pull"
        ],
        cwd=str(REPO),
        check=True
    )

else:

    print("Cloning official repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--recurse-submodules",
            "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git",
            str(REPO)
        ],
        check=True
    )

subprocess.run(
    [
        "git",
        "submodule",
        "update",
        "--init",
        "--recursive"
    ],
    cwd=str(REPO),
    check=True
)

print()
print("REPO:", REPO)

Repository exists.

REPO: /content/Hunyuan3D-2.1


In [7]:
# ============================================================
# CELL 4 — PYTORCH + SHAPE DEPENDENCIES
# ============================================================

import subprocess
from pathlib import Path

PYTHON = Path(
    "/content/hunyuan21_env/bin/python"
)

REPO = Path(
    "/content/Hunyuan3D-2.1"
)

print("=" * 70)
print("INSTALLING PYTORCH")
print("=" * 70)

# Official Tencent tested stack:
# torch 2.5.1 + cu124
# torchvision 0.20.1
# torchaudio 2.5.1

subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "torch==2.5.1",
        "torchvision==0.20.1",
        "torchaudio==2.5.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu124"
    ],
    check=True
)

print("=" * 70)
print("INSTALLING OFFICIAL SHAPE REQUIREMENTS")
print("=" * 70)

REQ = (
    REPO
    / "hy3dshape"
    / "requirements.txt"
)

subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "-r",
        str(REQ)
    ],
    check=True
)

# Missing import encountered in current source:
# timm.models.vision_transformer
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "timm",
        "huggingface_hub",
        "psutil"
    ],
    check=True
)

print("\nDependencies installed.")

INSTALLING PYTORCH
INSTALLING OFFICIAL SHAPE REQUIREMENTS

Dependencies installed.


In [ ]:
# ============================================================
# CELL 5 — VERIFY ENVIRONMENT
# ============================================================

import subprocess

PYTHON = "/content/hunyuan21_env/bin/python"

test = r'''
import sys
import torch

print("=" * 70)
print("PYTHON")
print("=" * 70)
print(sys.version)

print()
print("=" * 70)
print("PYTORCH")
print("=" * 70)

print("Version:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)

        print(
            f"GPU {i}: {p.name}"
        )

        print(
            "VRAM:",
            round(
                p.total_memory / 1024**3,
                2
            ),
            "GB"
        )

print()
print("=" * 70)
print("DEPENDENCIES")
print("=" * 70)

import timm
print("timm:", timm.__version__)

import rembg
print("rembg: OK")

import diffusers
print("diffusers:", diffusers.__version__)

import transformers
print("transformers:", transformers.__version__)

print()
print("=" * 70)
print("ALL BASIC IMPORTS: OK")
print("=" * 70)
'''

result = subprocess.run(
    [
        PYTHON,
        "-c",
        test
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout
)

if result.stderr:
    print(
        result.stderr
    )

if result.returncode != 0:
    raise RuntimeError(
        "Environment verification failed."
    )

In [ ]:
# ============================================================
# CELL 6 — PATCH HUNYUAN CHECKPOINT LOADER FOR LOW RAM
# ============================================================

from pathlib import Path

PIPELINE_FILE = Path(
    "/content/Hunyuan3D-2.1"
    "/hy3dshape/hy3dshape/pipelines.py"
)

text = PIPELINE_FILE.read_text()

old = (
    "ckpt = torch.load("
    "ckpt_path, map_location='cpu', weights_only=True"
    ")"
)

new = (
    "ckpt = torch.load("
    "ckpt_path, "
    "map_location='cpu', "
    "weights_only=True, "
    "mmap=True"
    ")"
)

if old in text:

    text = text.replace(
        old,
        new,
        1
    )

    # Add GC import
    if "import gc\n" not in text:

        text = text.replace(
            "import copy\n",
            "import copy\nimport gc\n",
            1
        )

    # Release checkpoint after state_dict copies.
    old_block = (
        "if 'conditioner' in ckpt:\n"
        "            conditioner.load_state_dict(ckpt['conditioner'])\n"
        "        image_processor"
    )

    new_block = (
        "if 'conditioner' in ckpt:\n"
        "            conditioner.load_state_dict(ckpt['conditioner'])\n"
        "\n"
        "        # Release CPU checkpoint before constructing the pipeline.\n"
        "        del ckpt\n"
        "        gc.collect()\n"
        "\n"
        "        image_processor"
    )

    if old_block in text:

        text = text.replace(
            old_block,
            new_block,
            1
        )

    PIPELINE_FILE.write_text(
        text
    )

    print(
        "Loader patched: mmap=True + ckpt cleanup"
    )

else:

    if "mmap=True" in text:

        print(
            "Loader already patched."
        )

    else:

        raise RuntimeError(
            "Expected torch.load() line was not found. "
            "Source revision changed."
        )

In [ ]:
# ============================================================
# CELL 7 — DOWNLOAD SHAPE CHECKPOINT
# ============================================================

from pathlib import Path
import shutil
import subprocess

PYTHON = "/content/hunyuan21_env/bin/python"

MODEL_ROOT = Path(
    "/root/.cache/hy3dgen/tencent/Hunyuan3D-2.1"
)

DIT_DIR = (
    MODEL_ROOT
    / "hunyuan3d-dit-v2-1"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ลบเฉพาะ incomplete DiT cache
if DIT_DIR.exists():

    print(
        "Removing old/incomplete DiT cache..."
    )

    shutil.rmtree(
        DIT_DIR,
        ignore_errors=True
    )

print("=" * 70)
print("DOWNLOADING SHAPE MODEL")
print("=" * 70)

code = r'''
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="tencent/Hunyuan3D-2.1",
    allow_patterns=[
        "hunyuan3d-dit-v2-1/*"
    ],
    local_dir="/root/.cache/hy3dgen/tencent/Hunyuan3D-2.1"
)

print("Downloaded:", path)
'''

subprocess.run(
    [
        PYTHON,
        "-c",
        code
    ],
    check=True
)

print()
print("=" * 70)
print("CHECKPOINT FILES")
print("=" * 70)

config = (
    DIT_DIR
    / "config.yaml"
)

ckpt = (
    DIT_DIR
    / "model.fp16.ckpt"
)

print(
    "config:",
    config.exists()
)

print(
    "checkpoint:",
    ckpt.exists()
)

if ckpt.exists():

    print(
        "size:",
        round(
            ckpt.stat().st_size / 1024**3,
            2
        ),
        "GB"
    )

if not ckpt.exists():

    raise RuntimeError(
        "model.fp16.ckpt was not downloaded."
    )

In [ ]:
# ============================================================
# CELL 8 — LOAD HUNYUAN3D-2.1 MODEL
# USE /content/hunyuan21_env/bin/python
# ============================================================

import subprocess
from pathlib import Path

PYTHON = "/content/hunyuan21_env/bin/python"
REPO = "/content/Hunyuan3D-2.1"

print("=" * 70)
print("CELL 8 — HUNYUAN3D MODEL LOAD")
print("=" * 70)

# ------------------------------------------------------------
# Test dependencies INSIDE the venv
# ------------------------------------------------------------

test_code = r'''
import sys

print("Python:", sys.version)

import torch
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import trimesh
print("trimesh: OK")

import yaml
print("pyyaml: OK")

import timm
print("timm: OK")

import rembg
print("rembg: OK")

import sys
sys.path.insert(
    0,
    "/content/Hunyuan3D-2.1/hy3dshape"
)

from hy3dshape.pipelines import (
    Hunyuan3DDiTFlowMatchingPipeline
)

print("Hunyuan pipeline import: OK")
'''

result = subprocess.run(
    [
        PYTHON,
        "-c",
        test_code
    ],
    cwd=REPO,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Hunyuan3D environment test failed."
    )


# ------------------------------------------------------------
# Load model in the venv
# ------------------------------------------------------------

load_code = r'''
import sys
import gc
import os
import psutil
import torch

sys.path.insert(
    0,
    "/content/Hunyuan3D-2.1/hy3dshape"
)

from hy3dshape.pipelines import (
    Hunyuan3DDiTFlowMatchingPipeline
)

print("=" * 70)
print("LOADING HUNYUAN3D-2.1")
print("=" * 70)

ram = psutil.virtual_memory()

print(
    "RAM available before:",
    round(ram.available / 1024**3, 2),
    "GB"
)

print(
    "VRAM allocated before:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

pipeline = (
    Hunyuan3DDiTFlowMatchingPipeline
    .from_pretrained(
        "tencent/Hunyuan3D-2.1",
        device="cuda",
        dtype=torch.float16
    )
)

print("MODEL LOAD OK")

# ------------------------------------------------------------
# CPU offload if supported
# ------------------------------------------------------------

try:

    pipeline.enable_model_cpu_offload(
        gpu_id=0
    )

    print(
        "CPU offload: ENABLED"
    )

except Exception as e:

    print(
        "CPU offload unavailable:",
        repr(e)
    )

# ------------------------------------------------------------
# Save pipeline so later cells can use it
# ------------------------------------------------------------

import pickle

# Pipeline is NOT pickled.
# Instead expose it through a small Python process API.
#
# The model remains resident in this subprocess.
# ------------------------------------------------------------

ram = psutil.virtual_memory()

print(
    "RAM available after:",
    round(ram.available / 1024**3, 2),
    "GB"
)

print(
    "VRAM allocated after:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("=" * 70)
print("HUNYUAN3D READY")
print("=" * 70)

# Keep process alive.
# This process owns the GPU model.
while True:
    import time
    time.sleep(3600)
'''

# ------------------------------------------------------------
# Start persistent model process
# ------------------------------------------------------------

MODEL_PROCESS = subprocess.Popen(
    [
        PYTHON,
        "-u",
        "-c",
        load_code
    ],
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print(
    "Started Hunyuan model process:",
    MODEL_PROCESS.pid
)

print()
print("Waiting for model process...")

# Read startup output until READY
ready = False

while True:

    line = MODEL_PROCESS.stdout.readline()

    if not line:
        break

    print(
        line.rstrip()
    )

    if "HUNYUAN3D READY" in line:

        ready = True
        break

    if MODEL_PROCESS.poll() is not None:
        break


if not ready:

    raise RuntimeError(
        "Hunyuan3D model process failed to start."
    )

print()
print("=" * 70)
print("CELL 8 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — IMAGE -> GLB TEST
# ============================================================

from pathlib import Path
from PIL import Image
import torch

REPO = Path(
    "/content/Hunyuan3D-2.1"
)

INPUT = (
    REPO
    / "assets"
    / "demo.png"
)

OUTPUT = (
    Path("/content")
    / "hunyuan_test.glb"
)

if not INPUT.exists():

    raise FileNotFoundError(
        f"Demo image not found: {INPUT}"
    )

print(
    "Input:",
    INPUT
)

print(
    "Output:",
    OUTPUT
)

image = Image.open(
    INPUT
).convert("RGBA")

print(
    "Generating..."
)

with torch.inference_mode():

    mesh = pipeline(
        image=image,

        # Official pipeline defaults are larger.
        # Conservative values for T4 16GB.
        num_inference_steps=5,
        guidance_scale=5.0,
        octree_resolution=256,
        num_chunks=8000,

        output_type="trimesh",

        enable_pbar=True
    )[0]


mesh.export(
    str(OUTPUT)
)

del mesh

torch.cuda.empty_cache()

print()
print("=" * 70)
print("SUCCESS")
print("=" * 70)

print(
    "GLB:",
    OUTPUT
)

print(
    "Size:",
    round(
        OUTPUT.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

In [ ]:
# ============================================================
# CELL 10 — HEADLESS API + NGROK
# ============================================================

import os
import sys
import uuid
import threading
import traceback
import time
from pathlib import Path

import torch
from PIL import Image

from fastapi import (
    FastAPI,
    UploadFile,
    File,
    HTTPException
)

from fastapi.responses import FileResponse

import uvicorn


# ============================================================
# CONFIG
# ============================================================

PORT = 8000

NGROK_DOMAIN = (
    "jaybird-heroic-garfish.ngrok-free.app"
)

NGROK_TOKEN = (
    "ใส่_TOKEN_ของคุณ"
)


# ============================================================
# DIRECTORIES
# ============================================================

BASE = Path(
    "/content/hunyuan21_api"
)

UPLOADS = BASE / "uploads"
OUTPUTS = BASE / "outputs"

UPLOADS.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUTS.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# API STATE
# ============================================================

jobs = {}

jobs_lock = threading.Lock()

gpu_lock = threading.Lock()


def update_job(
    job_id,
    **data
):

    with jobs_lock:

        jobs.setdefault(
            job_id,
            {}
        )

        jobs[job_id].update(
            data
        )


def read_job(
    job_id
):

    with jobs_lock:

        return dict(
            jobs.get(
                job_id,
                {}
            )
        )


# ============================================================
# API
# ============================================================

app = FastAPI(
    title="Hunyuan3D-2.1 Headless"
)


@app.get("/")
def root():

    return {
        "status": "online",
        "model":
            "tencent/Hunyuan3D-2.1",
        "type":
            "image-to-3d",
        "endpoint":
            "/generate"
    }


@app.get("/health")
def health():

    return {
        "status": "ok",
        "cuda":
            torch.cuda.is_available(),
        "gpu":
            (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            )
    }


@app.post("/generate")
async def generate(
    file: UploadFile = File(...)
):

    if not file.filename:

        raise HTTPException(
            status_code=400,
            detail="Filename missing"
        )

    filename = Path(
        file.filename
    ).name

    job_id = uuid.uuid4().hex[:12]

    input_path = (
        UPLOADS
        / f"{job_id}_{filename}"
    )

    output_name = (
        Path(filename).stem
        or f"model_{job_id}"
    )

    output_path = (
        OUTPUTS
        / f"{output_name}.glb"
    )

    data = await file.read()

    input_path.write_bytes(
        data
    )

    update_job(
        job_id,
        status="queued",
        filename=output_path.name
    )

    threading.Thread(
        target=process_job,
        args=(
            job_id,
            input_path,
            output_path
        ),
        daemon=True
    ).start()

    return {
        "job_id": job_id,
        "status": "queued",
        "filename": output_path.name
    }


def process_job(
    job_id,
    input_path,
    output_path
):

    try:

        with gpu_lock:

            update_job(
                job_id,
                status="loading"
            )

            image = Image.open(
                input_path
            ).convert("RGBA")

            update_job(
                job_id,
                status="generating"
            )

            with torch.inference_mode():

                mesh = pipeline(

                    image=image,

                    num_inference_steps=5,

                    guidance_scale=5.0,

                    octree_resolution=256,

                    num_chunks=8000,

                    output_type="trimesh",

                    enable_pbar=False

                )[0]


            update_job(
                job_id,
                status="exporting"
            )

            mesh.export(
                str(output_path)
            )

            del mesh

            torch.cuda.empty_cache()

            update_job(
                job_id,

                status="completed",

                output=str(
                    output_path
                ),

                filename=output_path.name
            )


    except Exception as e:

        traceback.print_exc()

        update_job(
            job_id,

            status="error",

            error=str(e),

            traceback=traceback.format_exc()
        )

        try:
            torch.cuda.empty_cache()
        except:
            pass

    finally:

        try:
            input_path.unlink()
        except:
            pass


@app.get(
    "/status/{job_id}"
)
def status(
    job_id: str
):

    data = read_job(
        job_id
    )

    if not data:

        raise HTTPException(
            status_code=404,
            detail="Job not found"
        )

    return data


@app.get(
    "/result/{job_id}"
)
def result(
    job_id: str
):

    data = read_job(
        job_id
    )

    if not data:

        raise HTTPException(
            status_code=404,
            detail="Job not found"
        )

    if data.get(
        "status"
    ) != "completed":

        raise HTTPException(
            status_code=409,
            detail=data
        )

    return FileResponse(
        data["output"],
        media_type="model/gltf-binary",
        filename=data["filename"]
    )


# ============================================================
# UVICORN
# ============================================================

def run_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()


# ============================================================
# WAIT
# ============================================================

import requests

for _ in range(120):

    try:

        r = requests.get(
            f"http://127.0.0.1:{PORT}/health",
            timeout=2
        )

        if r.status_code == 200:
            break

    except:
        pass

    time.sleep(1)

else:

    raise RuntimeError(
        "API failed to start"
    )


print(
    "API READY"
)


# ============================================================
# NGROK
# ============================================================

if (
    not NGROK_TOKEN
    or
    NGROK_TOKEN == "2nmFL8KW47qZ2cLOKfpYSHvvbmT_jEBbtqqq1hHCZWHW1ifk"
):

    raise RuntimeError(
        "ใส่ NGROK_TOKEN ก่อน"
    )


from pyngrok import ngrok

try:
    ngrok.kill()
except:
    pass

ngrok.set_auth_token(
    NGROK_TOKEN
)

tunnel = ngrok.connect(
    addr=PORT,
    proto="http",
    domain=NGROK_DOMAIN
)

PUBLIC_URL = str(
    tunnel.public_url
)

if PUBLIC_URL.startswith(
    "http://"
):

    PUBLIC_URL = (
        "https://"
        + PUBLIC_URL[7:]
    )


print()
print("=" * 70)
print("HUNYUAN3D HEADLESS ONLINE")
print("=" * 70)

print(
    "URL:",
    PUBLIC_URL
)

print(
    "HEALTH:",
    PUBLIC_URL + "/health"
)

print(
    "GENERATE:",
    PUBLIC_URL + "/generate"
)

print(
    "STATUS:",
    PUBLIC_URL + "/status/{job_id}"
)

print(
    "RESULT:",
    PUBLIC_URL + "/result/{job_id}"
)

print("=" * 70)


# ============================================================
# KEEP ALIVE
# ============================================================

while True:

    time.sleep(30)

    if api_thread.is_alive():

        print(
            "[HUNYUAN3D] ONLINE"
        )

    else:

        print(
            "[HUNYUAN3D] API THREAD STOPPED"
        )

        break